# NN_00 — setup and provenance

First notebook of the `neural_ranker/` series (`BUILD_SPEC.md` §7, row 00). It pins down what every later notebook runs against — interpreter, `torch` and device, the SHA-256 identity of the real Bloomberg exports, the strategy config identity, the cached real-data slice, and the seed policy — and asserts this row's acceptance criteria: **real SHA-256 digests are printed** and **the device resolves to `mps` or `cpu`**.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for `multi_factor_ranking_ml`, read through the strategy's own `_real_data.py` helper — the only sanctioned path (`BUILD_SPEC.md` §2). There is no synthetic fallback anywhere in this series: if the exports are absent, `rd.require_data()` prints an explicit *cannot produce a genuine result* notice and every computing cell is skipped. Nothing is approximated, backfilled, or simulated.

This notebook builds no tensors and trains nothing. It exists so that every number in `NN_01`–`NN_09` can be traced back to a recorded environment.

Kernel: `atlasquant-venv` (= `.venv/bin/python` at the repo root). The user-level `python3` kernelspec on this machine points at a *different* project's venv, so the series pins the kernel explicitly.

In [1]:
import nn_common as nc

rd = nc.rd  # the strategy's real-data helper, re-exported by nn_common
AVAILABLE = nc.provenance_banner("NN_00_setup_and_provenance.ipynb")

neural_ranker / NN_00_setup_and_provenance.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:4

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## Path setup

`nn_common.py` derives every path from its own file location — never from the kernel's working directory — and puts the strategy's notebook directory on `sys.path` so `_real_data` imports. `atlas_quant` itself is the editable install in `.venv`, pointing at this checkout's `src/`. Nothing under `src/atlas_quant/` is modified by this series (`BUILD_SPEC.md` §0, rule 4).

In [2]:
import atlas_quant

print("neural_ranker dir      ", nc.NEURAL_RANKER_DIR)
print("repo root              ", nc.REPO_ROOT)
print("strategy notebooks dir ", nc.STRATEGY_NOTEBOOKS_DIR)
print("_real_data module      ", rd.__file__)
print("atlas_quant package    ", atlas_quant.__file__)

assert (nc.REPO_ROOT / "CLAUDE.md").is_file(), "repo root did not resolve to the AtlasQuant checkout"
assert rd.REPO_ROOT == nc.REPO_ROOT, "nn_common and _real_data disagree on the repo root"
assert atlas_quant.__file__.startswith(str(nc.REPO_ROOT / "src")), "atlas_quant is not this checkout's src/ install"

neural_ranker dir       /Users/gavinhussey/Downloads/AtlasQuant/neural_ranker
repo root               /Users/gavinhussey/Downloads/AtlasQuant
strategy notebooks dir  /Users/gavinhussey/Downloads/AtlasQuant/research/strategies/multi_factor_ranking_ml/notebooks
_real_data module       /Users/gavinhussey/Downloads/AtlasQuant/research/strategies/multi_factor_ranking_ml/notebooks/_real_data.py
atlas_quant package     /Users/gavinhussey/Downloads/AtlasQuant/src/atlas_quant/__init__.py


## torch and device

`BUILD_SPEC.md` §1: `torch` 2.13.0, MPS with a CPU fallback, never assume CUDA. `nc.resolve_device()` is the single place the device is chosen; the probe below actually executes an op on it, so "resolves to `mps`" means "runs on `mps`", not merely "reports it".

In [3]:
import torch

DEVICE = nc.resolve_device()
print("torch version   ", torch.__version__)
print("mps built       ", torch.backends.mps.is_built())
print("mps available   ", torch.backends.mps.is_available())
print("cuda available  ", torch.cuda.is_available(), "(not used; never assumed)")
print("resolved device ", DEVICE)

probe = torch.arange(6, dtype=torch.float32, device=DEVICE).reshape(2, 3)
print("probe tensor    ", probe.device, tuple(probe.shape), "sum =", probe.sum().item())

# Acceptance (BUILD_SPEC §7, row 00): the device resolves to mps or cpu.
assert DEVICE.type in ("mps", "cpu"), f"unexpected device {DEVICE}"
assert probe.device.type == DEVICE.type and probe.sum().item() == 15.0

torch version    2.13.0
mps built        True
mps available    True
cuda available   False (not used; never assumed)
resolved device  mps
probe tensor     mps:0 (2, 3) sum = 15.0


## Real-data availability

`rd.data_available()` is true only when the gitignored raw root holds `fundamentals_quarterly.csv`. `rd.require_data()` is the guard every computing cell in this series sits behind: it prints the explicit unavailability notice and returns `False` when the data is absent, so a checkout without the data still executes top to bottom and produces *no number at all* rather than a fabricated one.

The manifest below hashes every CSV the pipeline reads. The same digests feed the slice cache's fingerprint, so a changed source file forces a rebuild rather than a silent reuse.

In [4]:
if rd.require_data():
    manifest = rd.source_manifest()
    print(f"raw root       {rd.RAW_ROOT}")
    print(f"required file  {rd.SOURCE_FILES[0]}  (the only strictly required export)")
    print()
    print(f"{'file':<28} {'present':>7} {'size (MB)':>10}  sha256")
    for entry in manifest:
        size = f"{entry.size_bytes / 1e6:.1f}" if entry.present else "-"
        print(f"{entry.name:<28} {str(entry.present):>7} {size:>10}  {entry.sha256 or '-'}")

    # Acceptance (BUILD_SPEC §7, row 00): real SHA-256 digests are printed.
    assert all(e.present for e in manifest), "a source CSV is absent"
    assert all(e.sha256 and len(e.sha256) == 64 and int(e.sha256, 16) >= 0 for e in manifest)
    assert len({e.sha256 for e in manifest}) == len(manifest), "two source files hash identically"

raw root       /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
required file  fundamentals_quarterly.csv  (the only strictly required export)

file                         present  size (MB)  sha256
fundamentals_quarterly.csv      True      281.3  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
Close_Price.csv                 True       67.6  2723e56e47e8671b7ffe65e856470a652af034fe2c181815d6ca5dcbc594b08b
Daily_Macro.csv                 True        0.6  b838ea58ba52393d35387d12bdbc8ece056f62ab0ded68bfddca9311b746a8dd
Fed_Funds_Rate.csv              True        0.2  b277d4a8d7fba7d03e95d6dcb686e33c2d773d128876fe610f825fdead2611f7


## Strategy identity

The config every later notebook builds against, with the defaults `BUILD_SPEC.md` §2 quotes asserted rather than trusted. One fact here matters directly for `NN_01`/`NN_02`: the feature pipeline **already** replaces the five features in `cross_sectional_rank_features` with their within-cycle percentile rank, so those five arrive in the cached slice as values in `[0, 1]`, not raw levels.

In [5]:
from atlas_quant.strategies.multi_factor_ranking_ml.config import (
    FEATURE_SCHEMA_VERSION,
    STRATEGY_ID,
    STRATEGY_VERSION,
)
from atlas_quant.strategies.multi_factor_ranking_ml.feature_domain import FEATURE_NAMES
from atlas_quant.strategies.multi_factor_ranking_ml.sector_encoding import SectorEncoder

cfg = rd.config()
print("strategy_id             ", STRATEGY_ID)
print("strategy_version        ", STRATEGY_VERSION)
print("feature_schema_version  ", FEATURE_SCHEMA_VERSION)
print("feature count           ", len(FEATURE_NAMES))
print("config identity         ", cfg.identity())
print("ml_train_years          ", cfg.ml_train_years)
print("min_train_quarters      ", cfg.min_train_quarters)
print("n_relevance_grades      ", cfg.n_relevance_grades)
print("return_cap              ", cfg.return_cap)
print("exclude_sectors         ", cfg.exclude_sectors)
print("already percentile-ranked by the feature pipeline (config.cross_sectional_rank_features):")
for name in cfg.cross_sectional_rank_features:
    print("    ", name)
vocabulary = SectorEncoder().consolidated_vocabulary()
print("sector vocabulary       ", len(vocabulary), vocabulary)

# BUILD_SPEC §2 quotes these; assert them rather than trust the quote.
assert len(FEATURE_NAMES) == 71
assert (cfg.ml_train_years, cfg.min_train_quarters, cfg.n_relevance_grades, cfg.return_cap) == (6, 4, 10, 0.50)
assert len(vocabulary) == 10

strategy_id              multi_factor_ranking_ml
strategy_version         0.3.0
feature_schema_version   4
feature count            71
config identity          b1e8a3339eecf744d501175f07c965c0a59882485ed5dd8c5382f41f173f7d75
ml_train_years           6
min_train_quarters       4
n_relevance_grades       10
return_cap               0.5
exclude_sectors          ('Materials',)
already percentile-ranked by the feature pipeline (config.cross_sectional_rank_features):
     market_cap
     volume
     net_debt
     adjusted_net_debt
     analyst_target_price
sector vocabulary        10 ('Consumer', 'Energy', 'Financials', 'Health Care', 'Industrials', 'Materials', 'Real Estate', 'Tech & Media', 'Unknown', 'Utilities')


## The cached real-data slice

`BUILD_SPEC.md` §1: a cached slice exists for the current config and `rd.slice_bundle()` returns it; building from scratch costs minutes and is never needed here. The cache file name carries a fingerprint of the source SHA-256s, the strategy config identity, `STRATEGY_VERSION`, `FEATURE_SCHEMA_VERSION`, and the slice bounds, so this cell also proves the cache on disk is the one for *this* configuration and *these* source files.

In [6]:
if rd.require_data():
    import time

    cache_file = rd.cache_path()
    print("cache file    ", cache_file)
    print("cache present ", cache_file.is_file(), f"({cache_file.stat().st_size / 1e6:.1f} MB)" if cache_file.is_file() else "")
    assert cache_file.is_file(), "no cached slice for the current fingerprint -- slice_bundle() would rebuild for minutes"

    t0 = time.perf_counter()
    bundle = rd.slice_bundle()
    print(f"loaded in      {time.perf_counter() - t0:.1f}s")
    print("fingerprint   ", bundle.fingerprint)
    print("built at      ", bundle.built_at)
    print("cycles        ", len(bundle.cycles), f"{bundle.cycles[0].quarter_start} .. {bundle.cycles[-1].quarter_start}")
    print("universe      ", len(bundle.universe), "instruments")
    print("source rows   ", f"{bundle.total_fundamentals_rows:,} fundamentals, {bundle.total_price_rows:,} daily prices (untrimmed)")
    print("load issues   ", bundle.load_issue_count)

    cached = {e.name: e.sha256 for e in bundle.source_manifest}
    live = {e.name: e.sha256 for e in rd.source_manifest()}
    assert cached == live, "cached slice was built from different source files than the live manifest"
    assert bundle.fingerprint == cache_file.stem.removeprefix("slice-")
    print("cache built from exactly the files hashed in the banner:", cached == live)

cache file     /Users/gavinhussey/Downloads/AtlasQuant/data/cache/multi_factor_ranking_ml/notebook_slice/slice-168fedca9e7b334fdf7fb0e4b8a15cb2.pickle
cache present  True (72.1 MB)


loaded in      2.3s
fingerprint    168fedca9e7b334fdf7fb0e4b8a15cb2
built at       2026-09-08 14:58:58.237518
cycles         14 2020-10-01 .. 2024-01-01
universe       1520 instruments
source rows    156,420 fundamentals, 9,245,992 daily prices (untrimmed)
load issues    734
cache built from exactly the files hashed in the banner: True


### Per-cycle shape of the slice

The magnitudes `BUILD_SPEC.md` §3 quotes (~1,463 names/quarter, 1.03 missing features per row) were measured on the full history; this is what the bounded slice actually holds, cycle by cycle, alongside the baseline's training state and IC for the same cycle.

In [7]:
if rd.require_data():
    baseline_by_cycle = {r.cycle.quarter_start: r for r in bundle.ic_result.cycle_results}
    header = f"{'quarter_start':<14}{'obs':>6}{'rejected':>10}{'missing/row':>13}{'labeled':>9}  {'baseline training':<30}{'baseline IC':>12}"
    print(header)
    print("-" * len(header))
    total_obs = total_missing = 0
    for cycle in bundle.cycles:
        result = bundle.feature_results[cycle.quarter_start]
        n = len(result.observations)
        missing = sum(len(o.missing_features) for o in result.observations)
        total_obs += n
        total_missing += missing
        labeled = bundle.labeled_quarters.get(cycle.quarter_start, ())
        r = baseline_by_cycle[cycle.quarter_start]
        ic = f"{r.ic:+.4f}" if r.ic is not None else "-"
        print(
            f"{cycle.quarter_start.isoformat():<14}{n:>6}{len(result.rejected):>10}{missing / n:>13.3f}"
            f"{len(labeled):>9}  {r.training_state.value:<30}{ic:>12}"
        )
    print()
    print(f"mean names/quarter   {total_obs / len(bundle.cycles):.1f}")
    print(f"mean missing/row     {total_missing / total_obs:.3f}   (over all {total_obs:,} observation rows in the slice)")
    assert all(len(bundle.feature_results[c.quarter_start].observations) > 0 for c in bundle.cycles)

quarter_start    obs  rejected  missing/row  labeled  baseline training              baseline IC
------------------------------------------------------------------------------------------------
2020-10-01      1423        97        1.552     1423  skipped_insufficient_quarters            -
2021-01-01      1430        90        1.592     1430  skipped_insufficient_quarters            -
2021-04-01      1436        84        1.428     1436  skipped_insufficient_quarters            -
2021-07-01      1451        69        1.606     1451  skipped_insufficient_quarters            -
2021-10-01      1462        58        1.576     1462  skipped_insufficient_quarters            -
2022-01-01      1468        52        1.402     1468  trained                            -0.0261
2022-04-01      1469        51        1.295     1469  trained                            -0.1900
2022-07-01      1472        48        1.127     1472  trained                            +0.0618
2022-10-01      1474        46

### The baseline the NN will be compared against

`bundle.ic_result` is the LightGBM LambdaRank baseline run over exactly these cycles with exactly the IC measure the NN will be scored by (`multi_factor_ranking_runner.spearman_correlation`). `NN_05`–`NN_08` pair against it cycle by cycle. Per `BUILD_SPEC.md` §5 rule 5 the universe is survivorship-biased, so none of these absolute numbers is a live expectation — only the *paired* comparison is meaningful.

In [8]:
if rd.require_data():
    from atlas_quant.strategies.multi_factor_ranking_ml.model_training import TrainingState

    ic_result = bundle.ic_result
    trained = [r for r in ic_result.cycle_results if r.training_state == TrainingState.TRAINED]
    measured = [
        r for r in ic_result.cycle_results
        if r.ic is not None and r.scored_for_ic_count >= ic_result.min_scored_count
    ]
    print("baseline estimator      LGBMRanker(objective='lambdarank'); model config identity", cfg.model.identity())
    print("cycles                 ", ic_result.cycle_count)
    print("trained cycles         ", len(trained), "->", ", ".join(r.cycle.quarter_start.isoformat() for r in trained))
    print("measured ICs           ", ic_result.measured_cycle_count, "(the last trained cycle has no next quarter to realize against)")
    print(f"mean IC                 {ic_result.mean_ic:+.4f}")
    print(f"IC std                  {ic_result.ic_std:.4f}")
    print(f"IC information ratio    {ic_result.ic_information_ratio:.4f}")
    print(f"hit rate                {ic_result.hit_rate:.3f}")
    print(f"mean decile spread      {ic_result.mean_decile_spread:+.4f}")
    print(f"min_scored_count        {ic_result.min_scored_count}")
    print("per-cycle IC           ", ", ".join(f"{r.ic:+.4f}" for r in measured))
    assert len(measured) == ic_result.measured_cycle_count

baseline estimator      LGBMRanker(objective='lambdarank'); model config identity 93fb836b34f4a47e0ca6c9a768e5583cdb2d0f10c211865aa0e292669172ee81
cycles                  14
trained cycles          9 -> 2022-01-01, 2022-04-01, 2022-07-01, 2022-10-01, 2023-01-01, 2023-04-01, 2023-07-01, 2023-10-01, 2024-01-01
measured ICs            8 (the last trained cycle has no next quarter to realize against)
mean IC                 +0.0275
IC std                  0.0916
IC information ratio    0.3002
hit rate                0.750
mean decile spread      +0.0593
min_scored_count        30
per-cycle IC            -0.0261, -0.1900, +0.0618, +0.0583, +0.0708, +0.0335, +0.0877, +0.1240


## Seed policy

`BUILD_SPEC.md` §8: five seeds minimum, recorded alongside the config identity, and never a single-run number. The seed set is `nc.SEEDS`; `nc.seed_everything(seed)` seeds `random`, `numpy`, and `torch` (CPU and MPS). What is *measured* here, on the resolved device: the same seed reproduces the same initial weights and the same dropout-active forward pass bit for bit, and a different seed does not. This is a property of the environment, not a result — but every later notebook's claim of "same seeds, paired" rests on it.

In [9]:
import numpy as np

print("SEEDS           ", nc.SEEDS)
print("config identity ", cfg.identity())


def _probe(seed: int):
    # A small init + dropout-active forward pass on DEVICE under `seed`.
    nc.seed_everything(seed)
    layer = torch.nn.Sequential(
        torch.nn.Linear(8, 16), torch.nn.GELU(), torch.nn.Dropout(0.3), torch.nn.Linear(16, 1)
    ).to(DEVICE)
    layer.train()
    x = torch.randn(32, 8, device=DEVICE)
    y = layer(x)
    return [p.detach().cpu().numpy().copy() for p in layer.parameters()], y.detach().cpu().numpy()


w_a, y_a = _probe(nc.SEEDS[0])
w_b, y_b = _probe(nc.SEEDS[0])
w_c, _ = _probe(nc.SEEDS[1])
same_init = all(np.array_equal(a, b) for a, b in zip(w_a, w_b))
same_forward = np.array_equal(y_a, y_b)
different_init = not all(np.array_equal(a, c) for a, c in zip(w_a, w_c))
print(f"same seed  -> identical init weights on {DEVICE}:        {same_init}")
print(f"same seed  -> identical forward pass (dropout active):  {same_forward}")
print(f"other seed -> different init weights:                   {different_init}")
assert same_init and same_forward and different_init

SEEDS            (0, 1, 2, 3, 4)
config identity  b1e8a3339eecf744d501175f07c965c0a59882485ed5dd8c5382f41f173f7d75
same seed  -> identical init weights on mps:        True
same seed  -> identical forward pass (dropout active):  True
other seed -> different init weights:                   True


## Point-in-time check

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6: **this notebook computes no feature, label, or normalization statistic, so it introduces no lookahead path of its own.** What it *did* verify, on every row of the cached slice rather than by assumption:

1. every cycle's `cutoff` is strictly before its `quarter_start`;
2. every feature observation's `feature_timestamp` is on or before its `data_cutoff`, and that `data_cutoff` is the cycle's own cutoff — no observation carries data from after the day the ranking is made;
3. every label's `label_available_at` is strictly after the *entry date* of the quarter it grades — no relevance grade is knowable at the moment its own cycle is ranked;
4. the final cycle has no labels at all, because no forward return has been realized for it.

The normalization-scope audit (that cycle *t*'s tensors are unchanged by the presence or absence of cycles after *t*) is `NN_03`'s job and is not claimed here.

In [10]:
if rd.require_data():
    from datetime import datetime, time as _time

    for cycle in bundle.cycles:
        assert cycle.cutoff < cycle.quarter_start

    n_obs = 0
    for cycle in bundle.cycles:
        for obs in bundle.feature_results[cycle.quarter_start].observations:
            assert obs.feature_timestamp <= obs.data_cutoff.date()
            assert obs.data_cutoff.date() == cycle.cutoff
            n_obs += 1

    n_labels = 0
    for quarter_start, rows in bundle.labeled_quarters.items():
        entry = datetime.combine(quarter_start, _time.min)
        for row in rows:
            assert row.label_available_at > entry, "a relevance grade would be knowable at its own cycle's entry"
            n_labels += 1

    assert bundle.cycles[-1].quarter_start not in bundle.labeled_quarters

    print(f"cycles checked        {len(bundle.cycles):>6}   cutoff < quarter_start                          True")
    print(f"observations checked  {n_obs:>6,}   feature_timestamp <= data_cutoff == cycle cutoff  True")
    print(f"labels checked        {n_labels:>6,}   label_available_at > own cycle's entry date      True")
    print(f"final cycle           {bundle.cycles[-1].quarter_start}   carries no labels (nothing realized yet)    True")

cycles checked            14   cutoff < quarter_start                          True
observations checked  20,517   feature_timestamp <= data_cutoff == cycle cutoff  True
labels checked        19,023   label_available_at > own cycle's entry date      True
final cycle           2024-01-01   carries no labels (nothing realized yet)    True


## What this established

Every number below was printed by a cell above in this executed run; nothing here is quoted from the spec.

**Environment.** Python 3.14.5, `torch` 2.13.0 with MPS built and available; the resolved device is `mps` and the probe op executed on `mps:0`. numpy 2.5.2, pandas 3.0.5, scipy 1.18.0, lightgbm 4.7.0. macOS 15.7.1 arm64. Git commit `7b7f2681146b8ed0a65ca808697a6194296a781f`. Kernel `atlasquant-venv` = `.venv/bin/python`. Both acceptance criteria for this row hold: real SHA-256 digests were printed, and the device resolved to `mps`.

**Real data.** Present at `data/raw/multi_factor_ranking_ml/`, all four exports:

| file | size | SHA-256 |
|---|---:|---|
| `fundamentals_quarterly.csv` | 281.3 MB | `3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a` |
| `Close_Price.csv` | 67.6 MB | `2723e56e47e8671b7ffe65e856470a652af034fe2c181815d6ca5dcbc594b08b` |
| `Daily_Macro.csv` | 0.6 MB | `b838ea58ba52393d35387d12bdbc8ece056f62ab0ded68bfddca9311b746a8dd` |
| `Fed_Funds_Rate.csv` | 0.2 MB | `b277d4a8d7fba7d03e95d6dcb686e33c2d773d128876fe610f825fdead2611f7` |

**Strategy identity.** `multi_factor_ranking_ml` 0.3.0, feature schema 4, 71 features; config identity `b1e8a3339eecf744d501175f07c965c0a59882485ed5dd8c5382f41f173f7d75`, LightGBM model-config identity `93fb836b34f4a47e0ca6c9a768e5583cdb2d0f10c211865aa0e292669172ee81`. Defaults asserted: `ml_train_years=6`, `min_train_quarters=4`, `n_relevance_grades=10`, `return_cap=0.50`. Sector vocabulary has 10 categories. Five features (`market_cap`, `volume`, `net_debt`, `adjusted_net_debt`, `analyst_target_price`) already arrive percentile-ranked from the feature pipeline.

**Cached slice.** Fingerprint `168fedca9e7b334fdf7fb0e4b8a15cb2`, built 2026-09-08 14:58:58, 72.1 MB, loads in ~2 s, and was built from exactly the four files hashed above. 14 cycles, 2020-10-01 .. 2024-01-01; universe of 1,520 instruments; 156,420 fundamentals rows and 9,245,992 daily price rows in the untrimmed source; 734 loader issues recorded by the strategy's own loader (its data-validation notebook, not this series, is where they are examined).

**Slice shape, measured.** 20,517 observation rows; 1,423 to 1,494 names per cycle (mean 1,465.5). Missing features per row: **1.252 over the whole slice**, ranging from 0.965 (2023-04-01) to 1.606 (2021-07-01) per cycle. The 1.03/row that `BUILD_SPEC.md` §3 quotes is a full-history figure; within this slice only the 2023-07 to 2023-10 cycles sit near it (1.048, 1.033). `NN_01` therefore checks its mask against the exact per-row `missing_features` counts of the cycle it builds, and reports the rate rather than asserting ≈1.03 blind.

**Baseline to pair against.** 9 trained cycles (2022-01-01 .. 2024-01-01), 8 with a measured IC: −0.0261, −0.1900, +0.0618, +0.0583, +0.0708, +0.0335, +0.0877, +0.1240. Mean IC **+0.0275**, IC std 0.0916, IC-IR 0.3002, hit rate 0.750, mean decile spread +0.0593 (`min_scored_count` 30, every measured cycle scored ≥ 1,370 names). These are *not* the §9/§10 numbers (0.0148 over 22 cycles), which come from a longer run; `NN_05`–`NN_08` pair against these 8 cycles, because these are the cycles the cached slice holds. Per §5 rule 5, no absolute figure here is a live expectation.

**Seed policy.** `SEEDS = (0, 1, 2, 3, 4)`, recorded with the config identity above. On `mps`, the same seed reproduced bit-identical initial weights and a bit-identical dropout-active forward pass; a different seed did not. Every later notebook reports means over these five seeds with the per-seed values alongside.

**Point-in-time.** Checked, not assumed, on the cached slice: 14 cycles (cutoff < quarter_start), 20,517 observations (feature_timestamp ≤ data_cutoff = cycle cutoff), 19,023 labels (label_available_at > own entry date), and the final cycle carries no labels. This notebook computes no feature, label, or normalization statistic, so it introduces no lookahead path of its own; the normalization-scope audit is `NN_03`.

**What `NN_01` needs from here.** `rd.slice_bundle()` (cached, ~2 s); `feature_domain.FEATURE_NAMES` for column order; the §4 partition (63 stock-level continuous + 6 macro + 2 categorical = 71); `SectorEncoder().consolidated_vocabulary()` (10 categories, `sector_enc` is already the index into it); `quarter_num` as a 1–4 float on every row; `LabeledObservation.relevance` in 0..9 from `bundle.labeled_quarters`; and the fact that the five ranked features are already in `[0, 1]`.